# Colab bootstrap — NNDL Saliency Project

Esegui queste celle in ordine ogni volta che apri una nuova sessione Colab.
**Regola d'oro:** salva SEMPRE i checkpoint su Drive, non solo sul disco della VM — la VM viene distrutta alla disconnessione.

## 1. Verifica GPU
Runtime > Change runtime type > GPU, poi esegui questa cella.

In [1]:
!nvidia-smi

Tue Sep 15 13:35:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Monta Google Drive (per checkpoint persistenti)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_PROJECT_DIR = '/content/drive/MyDrive/nndl-saliency'
CHECKPOINT_DIR = f'{DRIVE_PROJECT_DIR}/checkpoints'

# Archivio persistente del dataset
DATA_ARCHIVE = f'{DRIVE_PROJECT_DIR}/salicon.tar'

# Dataset veloce usato dalla VM Colab
LOCAL_DATA_DIR = '/content/data_local'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print('Checkpoint dir:', CHECKPOINT_DIR)
print('Dataset archive:', DATA_ARCHIVE)
print('Local dataset:', LOCAL_DATA_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checkpoint dir: /content/drive/MyDrive/nndl-saliency/checkpoints
Dataset archive: /content/drive/MyDrive/nndl-saliency/salicon.tar
Local dataset: /content/data_local


## 3. Clona/aggiorna il repository
La branch scelta viene aggiornata da GitHub con `pull --ff-only`; se il checkout locale contiene modifiche incompatibili, la cella si ferma. Prima dei run finali usare la branch/commit congelati.

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = 'https://github.com/markbtz/Project_NN.git'
REPO_DIR = '/content/nndl-saliency'
REPO_BRANCH = 'main'  # cambiare solo per un test esplicito di una branch

if (Path(REPO_DIR) / '.git').exists():
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'switch', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only', 'origin', REPO_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, REPO_DIR], check=True)

subprocess.run(['git', '-C', REPO_DIR, 'log', '-1', '--oneline'], check=True)

%cd $REPO_DIR

## 4. Installa le dipendenze mancanti

In [4]:
!pip install -q -r requirements-colab.txt

## 5. Credenziali Kaggle (solo la prima volta / se non già su Drive)
Carica `kaggle.json` quando richiesto (Kaggle > Settings > Create New Token).

In [5]:
import os
kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
kaggle_json_drive = '/content/drive/MyDrive/nndl-saliency/kaggle.json'
if os.path.exists(kaggle_json_drive):
    !cp "$kaggle_json_drive" ~/.kaggle/kaggle.json
else:
    from google.colab import files
    uploaded = files.upload()  # carica kaggle.json
    !mv kaggle.json ~/.kaggle/kaggle.json
    !cp ~/.kaggle/kaggle.json "$kaggle_json_drive"  # salva su Drive per la prossima volta
!chmod 600 ~/.kaggle/kaggle.json

In [6]:
!kaggle datasets list -s salicon

ref                             title                      size  lastUpdated                 downloadCount  voteCount  usabilityRating  
------------------------------  -------------------  ----------  --------------------------  -------------  ---------  ---------------  
hughiephan/salicon-mini         Salicon Mini           42143415  2024-09-18 08:55:53.797000             94          0  0.75             
harshgupta2411/salicon          SALICON              2726456977  2024-04-03 15:47:09.520000             97          0  0.0              
sarbojit3bhattachary/cognitive  Cognitive            5603129354  2025-07-19 18:45:05.943000              3          1  0.125            
roshan401/salicon               saliency in context  4264405363  2024-04-12 06:40:17.790000            607          2  0.5625           


## 6. Prepara SALICON su Drive (solo la prima volta)

Questa sezione scarica SALICON direttamente sul disco locale veloce di Colab, esegue l'audit, crea un unico `salicon.tar` e lo salva su Google Drive. Se `salicon.tar` esiste già, non riscarica nulla.

In [7]:
import os
import shutil
import subprocess

if os.path.exists(DATA_ARCHIVE):
    print('Archivio SALICON già presente su Drive:')
    print(DATA_ARCHIVE)
else:
    print('Archivio non presente: preparo SALICON per la prima volta.')

    # Rimuove solo un'eventuale copia locale incompleta della VM Colab.
    if os.path.exists(LOCAL_DATA_DIR):
        shutil.rmtree(LOCAL_DATA_DIR)
    os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

    print('\n1/4 - Download da Kaggle sul disco locale...')
    subprocess.run([
        'python', 'scripts/download_salicon.py',
        '--output_dir', LOCAL_DATA_DIR
    ], check=True)

    print('\n2/4 - Audit del dataset...')
    subprocess.run([
        'python', 'scripts/audit_dataset.py',
        '--data_dir', LOCAL_DATA_DIR
    ], check=True)

    LOCAL_ARCHIVE = '/content/salicon.tar'
    if os.path.exists(LOCAL_ARCHIVE):
        os.remove(LOCAL_ARCHIVE)

    print('\n3/4 - Creazione di salicon.tar...')
    subprocess.run([
        'tar', '-cf', LOCAL_ARCHIVE,
        '-C', LOCAL_DATA_DIR, '.'
    ], check=True)

    print('\n4/4 - Copia del singolo archivio su Google Drive...')
    shutil.copy2(LOCAL_ARCHIVE, DATA_ARCHIVE)
    os.remove(LOCAL_ARCHIVE)

    print('\nFatto. Archivio persistente salvato in:')
    print(DATA_ARCHIVE)

Archivio non presente: preparo SALICON per la prima volta.

1/4 - Download da Kaggle sul disco locale...

2/4 - Audit del dataset...

3/4 - Creazione di salicon.tar...

4/4 - Copia del singolo archivio su Google Drive...

Fatto. Archivio persistente salvato in:
/content/drive/MyDrive/nndl-saliency/salicon.tar


In [8]:
# Verifica che l'archivio persistente esista su Drive
if os.path.exists(DATA_ARCHIVE):
    size_gb = os.path.getsize(DATA_ARCHIVE) / (1024**3)
    print(f'OK: {DATA_ARCHIVE} ({size_gb:.2f} GB)')
else:
    print('ATTENZIONE: salicon.tar non è ancora presente su Drive.')

OK: /content/drive/MyDrive/nndl-saliency/salicon.tar (4.01 GB)


## 6bis. Prepara il dataset locale (a ogni nuova sessione)

Nelle sessioni successive NON riscaricare SALICON da Kaggle. Questa cella copia un solo archivio da Drive e lo estrae sul disco locale veloce della VM.

In [ ]:
import os
import shutil
import subprocess
import time

LOCAL_ARCHIVE = '/content/salicon.tar'

def fix_images_layout():
    """Corregge automaticamente il vecchio layout images/images/{train,val,test}."""
    images_dir = os.path.join(LOCAL_DATA_DIR, 'images')
    nested_images_dir = os.path.join(images_dir, 'images')

    if not os.path.isdir(nested_images_dir):
        return

    print('Rilevato layout images/images: correzione automatica...')

    for split in ['train', 'val', 'test']:
        src = os.path.join(nested_images_dir, split)
        dst = os.path.join(images_dir, split)

        if os.path.isdir(src) and not os.path.exists(dst):
            shutil.move(src, dst)

    if os.path.isdir(nested_images_dir) and not os.listdir(nested_images_dir):
        os.rmdir(nested_images_dir)

    print('Layout immagini corretto.')

if os.path.exists(LOCAL_DATA_DIR):
    fix_images_layout()
    print(f'{LOCAL_DATA_DIR} già presente. Salto preparazione.')
else:
    if not os.path.exists(DATA_ARCHIVE):
        raise FileNotFoundError(
            f'Archivio non trovato su Drive: {DATA_ARCHIVE}\n'
            'Esegui prima la sezione 6.'
        )

    t0 = time.time()

    print('Copio salicon.tar da Drive al disco locale...')
    shutil.copy2(DATA_ARCHIVE, LOCAL_ARCHIVE)

    print('Estraggo SALICON...')
    os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
    subprocess.run([
        'tar', '-xf', LOCAL_ARCHIVE,
        '-C', LOCAL_DATA_DIR
    ], check=True)

    os.remove(LOCAL_ARCHIVE)

    # Compatibilità con il vecchio salicon.tar.
    fix_images_layout()

    print(f'Dataset pronto in {LOCAL_DATA_DIR} in {(time.time() - t0) / 60:.1f} minuti.')


/content/data_local già presente. Salto preparazione.


In [10]:
# Controllo finale della copia locale
!python scripts/audit_dataset.py --data_dir "$LOCAL_DATA_DIR"

Scansione di /content/data_local ...

CONTEGGIO FILE PER CATEGORIA (euristico, verificare a occhio)
  images                   : 20000
  density_maps             : 15000
  fixation_files           : 20000
  other_structured_files   : 0
  other_files              : 0

DOMANDA CRITICA: fixation coordinates disponibili?
  TROVATI 20000 file che sembrano fixation data.
  Esempi:
    /content/data_local/fixations/train/COCO_train2014_000000522862.mat
    /content/data_local/fixations/train/COCO_train2014_000000181909.mat
    /content/data_local/fixations/train/COCO_train2014_000000207880.mat
    /content/data_local/fixations/train/COCO_train2014_000000391480.mat
    /content/data_local/fixations/train/COCO_train2014_000000477226.mat
  -> Apri manualmente un file per confermare il formato (es. .mat con array Nx2).
  -> Se confermato: potete usare NSS e sAUC oltre a CC/SIM/KLD.

CORRISPONDENZA IMMAGINE <-> DENSITY MAP
  Coppie corrispondenti: 15000
  Immagini senza mappa:  5000
  Mappe senza 

### Routine per le sessioni future

Quando Colab assegna una nuova VM, esegui nell'ordine: **1 → 2 → 3 → 4 → 6bis → 7.1**.  
La **sezione 5** serve se occorrono le credenziali Kaggle; la **sezione 6** serve normalmente una sola volta, finché `salicon.tar` rimane su Drive.


## 7. Esperimenti (script del repository)
Queste celle orchestrano test, training, evaluation e figure; non duplicano la logica degli script. I run brevi di sviluppo non sono risultati finali. Usare `tuning` fino al freeze del protocollo; `internal_test` resta chiuso.

### 7.1 Preflight
Eseguire sul commit appena aggiornato, prima di training o evaluation.

In [ ]:
%cd $REPO_DIR
!python -m pytest -q

### 7.2 Training
Di default non parte nulla. Prima dei run ufficiali congelare commit, YAML, seed e budget; usare checkpoint persistenti su Drive. La lista contiene solo i modelli supportati oggi da `train.py`.

In [ ]:
import subprocess

RUN_TRAINING = False
TRAIN_EXPERIMENTS = ['B0', 'B1', 'M1']

if RUN_TRAINING:
    for experiment in TRAIN_EXPERIMENTS:
        subprocess.run([
            'python', 'scripts/train.py',
            '--experiment', experiment,
            '--data_dir', LOCAL_DATA_DIR,
            '--checkpoint_dir', CHECKPOINT_DIR,
        ], cwd=REPO_DIR, check=True)

### 7.3 Evaluation su tuning
Per ogni checkpoint produce CSV per-image e JSON in Drive. Non inserire qui `internal_test`: è riservato alla valutazione finale dopo il freeze.

In [ ]:
from pathlib import Path
import subprocess

RUN_EVALUATION = False
EVALUATION_DIR = str(Path(DRIVE_PROJECT_DIR) / 'evaluation')
CHECKPOINTS = {
    'B0': 'B0_center_map.pt',
    'B1': 'B1_best.pt',
    'M1': 'M1_best.pt',
}

if RUN_EVALUATION:
    for experiment, filename in CHECKPOINTS.items():
        checkpoint = Path(CHECKPOINT_DIR) / filename
        if not checkpoint.is_file():
            raise FileNotFoundError(checkpoint)
        subprocess.run([
            'python', 'scripts/evaluate.py',
            '--experiment', experiment,
            '--checkpoint_path', str(checkpoint),
            '--split', 'tuning',
            '--data_dir', LOCAL_DATA_DIR,
            '--results_dir', EVALUATION_DIR,
        ], cwd=REPO_DIR, check=True)

### 7.4 Confronto per immagine
Lo script esistente allinea i CSV per `image_id` e segnala ID mancanti o duplicati. Le differenze sono `right−left`; la cella stampa anche la differenza media per CC/SIM/KLD (senza intervalli di confidenza).

In [ ]:
import csv
from statistics import mean
from pathlib import Path
import subprocess

RUN_COMPARISON = False
PAIRS = [('B0', 'B1'), ('B1', 'M1')]

if RUN_COMPARISON:
    for left, right in PAIRS:
        left_csv = Path(EVALUATION_DIR) / left / 'tuning_per_image.csv'
        right_csv = Path(EVALUATION_DIR) / right / 'tuning_per_image.csv'
        output = Path(EVALUATION_DIR) / 'comparisons' / f'{right}_minus_{left}.csv'
        subprocess.run([
            'python', 'scripts/compare_evaluations.py',
            str(left_csv), str(right_csv),
            '--left-name', left, '--right-name', right,
            '--output', str(output),
        ], cwd=REPO_DIR, check=True)
        with output.open(newline='', encoding='utf-8') as file:
            rows = list(csv.DictReader(file))
        for metric in ('cc', 'sim', 'kld'):
            delta = mean(float(row[f'delta_{metric}']) for row in rows)
            print(f'{right}−{left}: mean Δ{metric.upper()} = {delta:+.4f}')

### 7.5 Figure qualitative semplici
Solo dopo l'evaluation di B1 e M1 sul **tuning**: seleziona in modo riproducibile il caso migliore e peggiore di M1 per CC rispetto alla ground truth, più la variazione CC più favorevole di M1 rispetto a B1. Mostra immagine, ground truth e B0/B1/M1. Sono esempi illustrativi estremi, non una stima della performance media. Ogni mappa viene riscalata solo per visualizzare la forma; i valori quantitativi sono nei CSV/JSON. La PNG viene salvata su Drive.

In [ ]:
RUN_FIGURES = False

if RUN_FIGURES:
    import math
    import matplotlib.pyplot as plt
    import torch
    from pathlib import Path
    from scripts.compare_evaluations import load_per_image_csv, align_by_image_id
    from src.config_utils import load_yaml_config
    from src.data.dataset import SaliconDataset
    from src.runtime import get_device
    from scripts.evaluate import load_model_for_evaluation

    evaluation_dir = Path(DRIVE_PROJECT_DIR) / 'evaluation'
    b1_scores = load_per_image_csv(evaluation_dir / 'B1' / 'tuning_per_image.csv')
    m1_scores = load_per_image_csv(evaluation_dir / 'M1' / 'tuning_per_image.csv')
    aligned = align_by_image_id(
        b1_scores, m1_scores, left_name='B1', right_name='M1',
    )
    if any(not math.isfinite(r[1]['cc']) or not math.isfinite(r[2]['cc']) for r in aligned):
        raise ValueError('CC non finito nei CSV di B1/M1.')
    rankings = [
        ('M1 migliore per CC', sorted(aligned, key=lambda r: (-r[2]['cc'], r[0]))),
        ('M1 peggiore per CC', sorted(aligned, key=lambda r: (r[2]['cc'], r[0]))),
        ('ΔCC più favorevole M1−B1', sorted(aligned, key=lambda r: (-(r[2]['cc'] - r[1]['cc']), r[0]))),
    ]
    selected, used_ids = [], set()
    for label, candidates in rankings:
        choice = next((r for r in candidates if r[0] not in used_ids), None)
        if choice is None:
            raise ValueError('Servono almeno tre image_id distinti nei CSV.')
        selected.append((label, choice[0], choice[2]['cc'], choice[2]['cc'] - choice[1]['cc']))
        used_ids.add(choice[0])
    print('Esempi selezionati dal tuning:', selected)

    data_cfg = load_yaml_config('configs/data.yaml')
    experiment_cfg = load_yaml_config('configs/experiments.yaml')['experiments']
    width, height = map(int, data_cfg['input_size'])
    dataset = SaliconDataset(
        data_dir=LOCAL_DATA_DIR, manifest_path='results/split_manifest.csv',
        split='tuning', input_size=(width, height),
        density_map_epsilon=float(data_cfg['density_map_epsilon']),
        augmentation=False,
    )
    index_by_id = {row['image_id']: i for i, row in enumerate(dataset.samples)}
    if set(index_by_id) != {r[0] for r in aligned}:
        raise ValueError('I CSV non coincidono con gli image_id del tuning corrente.')
    device = get_device()
    models = {}
    for experiment, filename in CHECKPOINTS.items():
        model, predict_fn, *_ = load_model_for_evaluation(
            experiment, experiment_cfg[experiment],
            str(Path(CHECKPOINT_DIR) / filename), device, height, width,
        )
        model.eval()
        models[experiment] = (model, predict_fn)

    def display_map(tensor):
        values = tensor.detach().float().cpu().squeeze().numpy()
        low, high = values.min(), values.max()
        return (values - low) / (high - low) if high > low else values * 0

    fig, axes = plt.subplots(len(selected), 5, figsize=(16, 3.6 * len(selected)), squeeze=False)
    titles = ['Immagine', 'Ground truth', 'B0', 'B1', 'M1']
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    for row, (label, image_id, cc_m1, delta_cc) in enumerate(selected):
        sample = dataset[index_by_id[image_id]]
        image = (sample['image'] * std + mean).clamp(0, 1)
        axes[row, 0].imshow(image.permute(1, 2, 0).numpy())
        axes[row, 1].imshow(display_map(sample['density_map_raw']), cmap='magma', vmin=0, vmax=1)
        for col, experiment in enumerate(CHECKPOINTS, start=2):
            model, predict_fn = models[experiment]
            batch = {'image': sample['image'].unsqueeze(0)}
            with torch.inference_mode():
                prediction = (predict_fn(model, batch, device) if predict_fn
                              else model(batch['image'].to(device)))
            axes[row, col].imshow(display_map(prediction), cmap='magma', vmin=0, vmax=1)
        for col, title in enumerate(titles):
            axes[row, col].set_title(
                f'{label}\n{image_id}\nCC M1={cc_m1:.3f}; ΔCC={delta_cc:+.3f}' if col == 0 else title,
                fontsize=9,
            )
            axes[row, col].axis('off')
    fig.tight_layout()
    output = Path(DRIVE_PROJECT_DIR) / 'figures' / 'tuning_best_worst_gain.png'
    output.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output, dpi=160, bbox_inches='tight')
    plt.show()
    print('Figura salvata in:', output)

### 7.6 Internal test finale
Nessuna cella eseguibile qui per ora. Solo dopo il freeze di codice, split, modelli e configurazioni, valutare una volta ogni checkpoint finale con `scripts/evaluate.py --split internal_test --final_evaluation`; salvare i risultati separati da quelli di tuning.